In [0]:
# QC를 데이터로 기록하는 단계 -> QC도 하나의 데이터 파이프라인 산출물로 만드는 것
# Silver Layer의 품질을 정기적으로 검증할 수 있도록
# 각 Silver Delta Table을 현재 Spark session으로 로드한다.

from pyspark.sql import functions as F
from datetime import datetime
import uuid

silver_studies_df = spark.read.table(
    "clinical_trials.silver_studies"
)

silver_conditions_df = spark.read.table(
    "clinical_trials.silver_conditions"
)

silver_interventions_df = spark.read.table(
    "clinical_trials.silver_interventions"
)

silver_outcomes_df = spark.read.table(
    "clinical_trials.silver_outcomes"
)

silver_locations_df = spark.read.table(
    "clinical_trials.silver_locations"
)

In [0]:
# 한 번의 QC 실행을 하나의 run으로 관리하기 위해 UUID를 생성한다.
# 동일한 run_id를 통해 같은 실행에서 수행된 여러 품질 검사를 묶어 추적할 수 있다.
run_id = str(uuid.uuid4())

print(run_id)

# QC 결과를 담을 공간
quality_results = []

# QC 결과 하나를 계속 모아주는 Python 함수
def add_quality_result(
    table_name,
    rule_name,
    checked_rows,
    failed_rows
):
    failure_rate = (
        failed_rows / checked_rows
        if checked_rows > 0
        else 0.0
    )

    quality_results.append({
        "run_id": run_id,
        "table_name": table_name,
        "rule_name": rule_name,
        "checked_rows": checked_rows,
        "failed_rows": failed_rows,
        "failure_rate": failure_rate,
        "executed_at": datetime.now()
    })

8e0d84dc-df3e-454b-817a-3b0987847b91


In [0]:
# QC Rule 1: NCT ID는 null 이면 안된다
# Study의 식별자가 없으면 다른 Condition / Intervention / Outcome 등과 연결할 수 없음

checked_rows = silver_studies_df.count()

failed_rows = (
    silver_studies_df
    .filter(F.col("nct_id").isNull())
    .count()
)

add_quality_result(
    "silver_studies",
    "nct_id_not_null",
    checked_rows,
    failed_rows
)

In [0]:
# QC Rule 2: NCT ID는 중복이면 안된다
# Study 단위 테이블에서 nct_id는 고유 식별자 역할을 하므로 중복된 nct_id가 존재하는지 검증한다.

checked_rows = silver_studies_df.count()

unique_rows = (
    silver_studies_df
    .select("nct_id")
    .dropDuplicates()
    .count()
)

failed_rows = checked_rows - unique_rows

add_quality_result(
    "silver_studies",
    "nct_id_unique",
    checked_rows,
    failed_rows
)

In [0]:
# QC Rule 3: Enrollment는 음수가 될 수 없다 - Domain Validation

checked_rows = silver_studies_df.count()

failed_rows = (
    silver_studies_df
    .filter(F.col("enrollment_count") < 0)
    .count()
)

add_quality_result(
    "silver_studies",
    "enrollment_non_negative",
    checked_rows,
    failed_rows
)

In [0]:
# QC Rule 4: 완료일이 시작일보다 빠르면 안 된다 - 시간적 정합성(temporal consistency) 검사

checked_rows = silver_studies_df.count()

failed_rows = (
    silver_studies_df
    .filter(
        F.col("start_date").isNotNull() &
        F.col("completion_date").isNotNull() &
        (F.col("completion_date") < F.col("start_date"))
    )
    .count()
)

add_quality_result(
    "silver_studies",
    "completion_date_after_start_date",
    checked_rows,
    failed_rows
)

In [0]:
# Child table의 nct_id가 부모 silver_studies에 존재하는지 검증한다.
# left_anti join을 이용해 부모가 없는 orphan record를 탐지한다.

def check_orphans(child_df, table_name):
    checked_rows = child_df.count()

    failed_rows = (
        child_df
        .join(
            silver_studies_df.select("nct_id"),
            on="nct_id",
            how="left_anti"
        )
        .count()
    )

    add_quality_result(
        table_name,
        "nct_id_exists_in_silver_studies",
        checked_rows,
        failed_rows
    )

In [0]:
# QC Rule 5: Referential Integrity

check_orphans(
    silver_conditions_df,
    "silver_conditions"
)

check_orphans(
    silver_interventions_df,
    "silver_interventions"
)

check_orphans(
    silver_outcomes_df,
    "silver_outcomes"
)

check_orphans(
    silver_locations_df,
    "silver_locations"
)

In [0]:
# QC Rule 6: 위도 범위 검사

checked_rows = silver_locations_df.count()

failed_rows = (
    silver_locations_df
    .filter(
        F.col("latitude").isNotNull() &
        (
            (F.col("latitude") < -90) |
            (F.col("latitude") > 90)
        )
    )
    .count()
)

add_quality_result(
    "silver_locations",
    "latitude_valid_range",
    checked_rows,
    failed_rows
)

In [0]:
# QC Rule 7: 경도 범위 검사
checked_rows = silver_locations_df.count()

failed_rows = (
    silver_locations_df
    .filter(
        F.col("longitude").isNotNull() &
        (
            (F.col("longitude") < -180) |
            (F.col("longitude") > 180)
        )
    )
    .count()
)

add_quality_result(
    "silver_locations",
    "longitude_valid_range",
    checked_rows,
    failed_rows
)

In [0]:
# Python List형태의 quality_result를 Spark DataFrame으로 변환
quality_results_df = spark.createDataFrame(
    quality_results
)

display(
    quality_results_df.orderBy(
        "table_name",
        "rule_name"
    )
)

# 실패한 Rule만 확인하기
display(
    quality_results_df
    .filter(F.col("failed_rows") > 0)
)

checked_rows,executed_at,failed_rows,failure_rate,rule_name,run_id,table_name
1087211,2026-09-26T12:19:10.923Z,0,0.0,nct_id_exists_in_silver_studies,8e0d84dc-df3e-454b-817a-3b0987847b91,silver_conditions
1022647,2026-09-26T12:19:12.341Z,0,0.0,nct_id_exists_in_silver_studies,8e0d84dc-df3e-454b-817a-3b0987847b91,silver_interventions
3532550,2026-09-26T12:19:16.996Z,0,0.0,latitude_valid_range,8e0d84dc-df3e-454b-817a-3b0987847b91,silver_locations
3532550,2026-09-26T12:19:18.060Z,0,0.0,longitude_valid_range,8e0d84dc-df3e-454b-817a-3b0987847b91,silver_locations
3532550,2026-09-26T12:19:15.723Z,0,0.0,nct_id_exists_in_silver_studies,8e0d84dc-df3e-454b-817a-3b0987847b91,silver_locations
3567302,2026-09-26T12:19:14.000Z,0,0.0,nct_id_exists_in_silver_studies,8e0d84dc-df3e-454b-817a-3b0987847b91,silver_outcomes
604566,2026-09-26T12:19:08.832Z,0,0.0,completion_date_after_start_date,8e0d84dc-df3e-454b-817a-3b0987847b91,silver_studies
604566,2026-09-26T12:19:07.144Z,0,0.0,enrollment_non_negative,8e0d84dc-df3e-454b-817a-3b0987847b91,silver_studies
604566,2026-09-26T12:19:04.164Z,0,0.0,nct_id_not_null,8e0d84dc-df3e-454b-817a-3b0987847b91,silver_studies
604566,2026-09-26T12:19:06.124Z,0,0.0,nct_id_unique,8e0d84dc-df3e-454b-817a-3b0987847b91,silver_studies


checked_rows,executed_at,failed_rows,failure_rate,rule_name,run_id,table_name


In [0]:
# QC 결과는 실행 이력을 누적해야 하므로 overwrite가 아니라 append 방식으로 저장한다.
# 이를 통해 run별 품질 변화와 장애 발생 시점을 추적할 수 있다.

# overwrite : 기존 데이터를 덮어쓰기
# append : 기존 데이터에 추가하기

(
    quality_results_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("clinical_trials.quality_results")
)

In [0]:
%sql
-- 저장된 QC History 확인

SELECT
    run_id,
    table_name,
    rule_name,
    checked_rows,
    failed_rows,
    ROUND(failure_rate * 100, 2) AS failure_rate_pct,
    executed_at
FROM clinical_trials.quality_results
ORDER BY executed_at DESC, table_name, rule_name;

run_id,table_name,rule_name,checked_rows,failed_rows,failure_rate_pct,executed_at
8e0d84dc-df3e-454b-817a-3b0987847b91,silver_locations,longitude_valid_range,3532550,0,0.0,2026-09-26T12:19:18.060Z
8e0d84dc-df3e-454b-817a-3b0987847b91,silver_locations,latitude_valid_range,3532550,0,0.0,2026-09-26T12:19:16.996Z
8e0d84dc-df3e-454b-817a-3b0987847b91,silver_locations,nct_id_exists_in_silver_studies,3532550,0,0.0,2026-09-26T12:19:15.723Z
8e0d84dc-df3e-454b-817a-3b0987847b91,silver_outcomes,nct_id_exists_in_silver_studies,3567302,0,0.0,2026-09-26T12:19:14.000Z
8e0d84dc-df3e-454b-817a-3b0987847b91,silver_interventions,nct_id_exists_in_silver_studies,1022647,0,0.0,2026-09-26T12:19:12.341Z
8e0d84dc-df3e-454b-817a-3b0987847b91,silver_conditions,nct_id_exists_in_silver_studies,1087211,0,0.0,2026-09-26T12:19:10.923Z
8e0d84dc-df3e-454b-817a-3b0987847b91,silver_studies,completion_date_after_start_date,604566,0,0.0,2026-09-26T12:19:08.832Z
8e0d84dc-df3e-454b-817a-3b0987847b91,silver_studies,enrollment_non_negative,604566,0,0.0,2026-09-26T12:19:07.144Z
8e0d84dc-df3e-454b-817a-3b0987847b91,silver_studies,nct_id_unique,604566,0,0.0,2026-09-26T12:19:06.124Z
8e0d84dc-df3e-454b-817a-3b0987847b91,silver_studies,nct_id_not_null,604566,0,0.0,2026-09-26T12:19:04.164Z
